In [ ]:
import cv2
import zxingcpp

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
print("Resolución real:", cap.get(3), "x", cap.get(4))


x1, y1 = 500, 150
x2, y2 = 1700, 800


while True:
    ret, frame = cap.read()

    # Dibujar recuadro verde
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imshow("Captura", frame)

    key = cv2.waitKey(1)

    # Presiona S para leer
    if key == ord("s"):

        # Recortar SOLO el área definida
        cropped = frame[y1:y2, x1:x2]

        # Convertir a RGB (ZXing recibe en RGB)
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

        # Leer códigos con ZXing
        results = zxingcpp.read_barcodes(cropped_rgb)

        if len(results) == 0:
            print("No se encontró ningún código PDF417 dentro del recuadro.")
        else:
            print("\n===== CÓDIGOS DETECTADOS =====")
            for r in results:
                print(f"Formato: {r.format}")
                print(f"Datos  : {r.text}\n")

    # ESC para salir
    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import zxingcpp
import re

# ===============================
# Función para limpiar y extraer datos de la cédula
# ===============================
def parse_pdf417(text):
    # Quitar caracteres no imprimibles
    clean = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', text)


    data = {}

    # ===============================
    # 1. Encontrar TODAS las cadenas numéricas de 10 dígitos
    # ===============================
    all_10_digits = re.findall(r'(?<!\d)\d{10}(?!\d)', clean)

    # Regla:
    # - Ignorar la primera
    # - Si existe segunda → esa es la cédula
    if len(all_10_digits) >= 2:
        data["cedula"] = all_10_digits[1]  # segunda coincidencia
    else:
        data["cedula"] = None

    # ===============================
    # 2. Ignorar cadenas de 6 dígitos (NO HACER NADA CON ELLAS)
    # (no se guardan, solo se ignoran)
    # ===============================
    # Patrón: \b\d{6}\b
    # Solo las omitimos y seguimos

    # ===============================
    # 3. Buscar sexo + fecha (MYYYYMMDD / FYYYYMMDD)
    # ===============================
    m = re.search(r'([MF])(\d{8})', clean)
    if m:
        data["sexo"] = m.group(1)
        data["fecha_nac"] = m.group(2)

    # ===============================
    # 4. Buscar RH
    # ===============================
    m = re.search(r'(A|B|O)[+-]', clean)
    if m:
        data["rh"] = m.group(0)

    # ===============================
    # 5. Extraer apellidos y nombre
    # ===============================
    m = re.search(r'(\d{10})([A-ZÑÁÉÍÓÚ]+)([A-ZÑÁÉÍÓÚ]+)([A-ZÑÁÉÍÓÚ]+).*?[MF]\d{8}', clean)
    if m:
        data["apellido1"] = m.group(2)
        data["apellido2"] = m.group(3)
        data["nombre"]   = m.group(4)

    return clean, data


# ===============================
# Configuración de cámara
# ===============================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
print("Resolución real:", cap.get(3), "x", cap.get(4))

# Recuadro
x1, y1 = 500, 150
x2, y2 = 1700, 800

# ===============================
# Bucle principal
# ===============================
while True:
    ret, frame = cap.read()

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imshow("Captura", frame)

    key = cv2.waitKey(1)

    if key == ord("s"):

        cropped = frame[y1:y2, x1:x2]
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

        results = zxingcpp.read_barcodes(cropped_rgb)

        if len(results) == 0:
            print("No se encontró ningún código PDF417 dentro del recuadro.")
        else:
            print("\n===== CÓDIGOS DETECTADOS =====")

            for r in results:
                print(f"Formato: {r.format}")

                # Limpieza y extracción
                clean, extracted = parse_pdf417(r.text)

                print("\n--- Texto limpio ---")
                print(clean)

                print("\n--- Datos extraídos ---")
                for key, val in extracted.items():
                    print(f"{key}: {val}")

                print("\n")

    if key == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import zxingcpp
import json

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
print("Resolución real:", cap.get(3), "x", cap.get(4))

# Recuadro
x1, y1 = 500, 150
x2, y2 = 1700, 800

while True:
    ret, frame = cap.read()

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imshow("Captura", frame)

    key = cv2.waitKey(1)

    if key == ord("s"):
        cropped = frame[y1:y2, x1:x2]
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

        results = zxingcpp.read_barcodes(cropped_rgb)

        if len(results) == 0:
            print("No se encontró ningún código en el recuadro.")
        else:
            print("\n===== CÓDIGOS DETECTADOS =====")

            for r in results:
                print(f"Formato: {r.format}")
                print(f"Datos  : {r.text}\n")

                # ---------------------------------------------------
                # Detectar PDF417
                # ---------------------------------------------------
                if r.format == zxingcpp.BarcodeFormat.PDF417:
                    print("→ Se detectó PDF417 de la cédula")

                # ---------------------------------------------------
                # Detectar QR
                # ---------------------------------------------------
                if r.format == zxingcpp.BarcodeFormat.QR_CODE:
                    print("→ Se detectó QR de la nueva cédula")

                    try:
                        qr_json = json.loads(r.text)
                        print("===== DATOS DEL QR =====")
                        for k, v in qr_json.items():
                            print(f"{k}: {v}")
                    except:
                        print("El QR no contiene JSON o está protegido/codificado")

    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import zxingcpp
import re

# ===============================
# Función para limpiar y extraer datos de la cédula
# ===============================
def parse_pdf417(text):
    # Quitar caracteres no imprimibles
    clean = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', text)
    clean = re.sub(r'\s+', ' ', clean)

    data = {}

    # ===============================
    # 1. Encontrar TODAS las cadenas numéricas de 10 dígitos
    # ===============================
    all_10_digits = re.findall(r'(?<!\d)\d{10}(?!\d)', clean)

    # Regla:
    # - Ignorar la primera
    # - Si existe segunda → esa es la cédula
    if len(all_10_digits) >= 2:
        data["cedula"] = all_10_digits[1]  # segunda coincidencia
    else:
        data["cedula"] = None

    # ===============================
    # 2. Ignorar cadenas de 6 dígitos (NO HACER NADA CON ELLAS)
    # (no se guardan, solo se ignoran)
    # ===============================
    # Patrón: \b\d{6}\b
    # Solo las omitimos y seguimos

    # ===============================
    # 3. Buscar sexo + fecha (MYYYYMMDD / FYYYYMMDD)
    # ===============================
    m = re.search(r'([MF])(\d{8})', clean)
    if m:
        data["sexo"] = m.group(1)
        data["fecha_nac"] = m.group(2)

    # ===============================
    # 4. Buscar RH
    # ===============================
    m = re.search(r'(A|B|O)[+-]', clean)
    if m:
        data["rh"] = m.group(0)

    # ===============================
    # 5. Extraer apellidos y nombre
    # ===============================
    m = re.search(r'\d{10}\s+([A-ZÑÁÉÍÓÚ]+)\s+([A-ZÑÁÉÍÓÚ]+)\s+([A-ZÑÁÉÍÓÚ]+)', clean)
    if m:
        data["apellido1"] = m.group(2)
        data["apellido2"] = m.group(3)
        data["nombre"]   = m.group(4)

    return clean, data


# ===============================
# Configuración de cámara
# ===============================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
print("Resolución real:", cap.get(3), "x", cap.get(4))

# Recuadro
x1, y1 = 500, 150
x2, y2 = 1700, 800

# ===============================
# Bucle principal
# ===============================
while True:
    ret, frame = cap.read()

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imshow("Captura", frame)

    key = cv2.waitKey(1)

    if key == ord("s"):

        cropped = frame[y1:y2, x1:x2]
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

        results = zxingcpp.read_barcodes(cropped_rgb)

        if len(results) == 0:
            print("No se encontró ningún código PDF417 dentro del recuadro.")
        else:
            print("\n===== CÓDIGOS DETECTADOS =====")

            for r in results:
                print(f"Formato: {r.format}")

                # Limpieza y extracción
                clean, extracted = parse_pdf417(r.text)

                print("\n--- Texto limpio ---")
                print(clean)

                print("\n--- Datos extraídos ---")
                for key, val in extracted.items():
                    print(f"{key}: {val}")

                print("\n")

    if key == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import zxingcpp
import re
import unicodedata



def limpiar(texto):
    # Quitar caracteres no ASCII imprimibles
    texto = ''.join(c for c in texto if 32 <= ord(c) <= 126)
    return texto

def extraer_datos_pdf417(raw):
    txt = limpiar(raw)

    # Buscar la cédula (10 dígitos)
    m = re.search(r'(\d{10})', txt)
    if not m:
        return None
    
    inicio = m.start()
    bloque = txt[inicio:]    # Tomamos desde la cédula hacia adelante

    # Longitudes en el PDF417 real
    long_cedula = 10
    long_ap1    = 15
    long_ap2    = 15
    long_nom    = 15

    cedula = bloque[0:10]

    ap1   = bloque[10:10+15].strip()
    ap2   = bloque[25:25+15].strip()
    nom   = bloque[40:40+15].strip()

    # Sexo, fecha, RH
    sexo = bloque[55]           # M / F
    fecha = bloque[56:64]       # YYYYMMDD
    rh = bloque[64:67].replace("<","").strip()

    return {
        "cedula": cedula,
        "apellido1": ap1,
        "apellido2": ap2,
        "nombre": nom,
        "sexo": sexo,
        "fecha_nac": fecha,
        "rh": rh
    }

# ===============================
# Configuración de cámara
# ===============================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
print("Resolución real:", cap.get(3), "x", cap.get(4))

# Recuadro
x1, y1 = 500, 150
x2, y2 = 1700, 800

# ===============================
# Bucle principal
# ===============================
while True:
    ret, frame = cap.read()

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imshow("Captura", frame)

    key = cv2.waitKey(1)

    if key == ord("s"):

        cropped = frame[y1:y2, x1:x2]
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

        results = zxingcpp.read_barcodes(cropped_rgb)

        if len(results) == 0:
            print("No se encontró ningún código PDF417 dentro del recuadro.")
        else:
            print("\n===== CÓDIGOS DETECTADOS =====")

            for r in results:
                print(f"Formato: {r.format}")

                # Limpieza y extracción
                clean, extracted = parse_pdf417(r.text)

                print("\n--- Texto limpio ---")
                print(clean)

                print("\n--- Datos extraídos ---")
                for key, val in extracted.items():
                    print(f"{key}: {val}")

                print("\n")

    if key == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()


SyntaxError: invalid syntax (3360474367.py, line 1)

In [3]:
import cv2
import zxingcpp
import re

# ===============================
# Función para limpiar y extraer datos de la cédula
# ===============================
def parse_pdf417(text):
    # Quitar caracteres no imprimibles
    clean = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', text)

    # Quitar palabras "NUL" que vienen del decodificador
    clean = clean.replace("NUL", " ")

    data = {}

    # ===============================
    # 1. Encontrar las cadenas numéricas de 10 dígitos
    # ===============================
    all_10_digits = re.findall(r'(?<!\d)\d{10}(?!\d)', clean)

    if len(all_10_digits) >= 2:
        cedula = all_10_digits[1]
        data["cedula"] = cedula
    else:
        data["cedula"] = None
        cedula = None

    # ===============================
    # 2. Sexo + fecha
    # ===============================
    m = re.search(r'([MF])(\d{8})', clean)
    if m:
        data["sexo"] = m.group(1)
        data["fecha_nac"] = m.group(2)
        

        

    # ===============================
    # 3. RH
    # ===============================
    m = re.search(r'(A|B|O)[+-]', clean)
    if m:
        data["rh"] = m.group(0)

    # ===============================
    # 4. Apellidos y nombre SIN “NUL”
    # ===============================
    if cedula:
        pos = clean.find(cedula)
        tail = clean[pos + len(cedula):]

        # Grupos reales de letras (mínimo 2 letras)
        grupos = re.findall(r'\b[A-ZÑÁÉÍÓÚ]{2,}\b', tail)

        # Evitar basura
        grupos = [g for g in grupos if g not in ["N", "NU", "NUL"]]

        if len(grupos) >= 1:
            data["apellido1"] = grupos[0]
        if len(grupos) >= 2:
            data["apellido2"] = grupos[1]
        if len(grupos) >= 3:
            data["nombre"] = grupos[2]
        
    
           

    return clean, data



# ===============================
# Configuración de cámara
# ===============================
cap = cv2.VideoCapture(2)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
print("Resolución real:", cap.get(3), "x", cap.get(4))

# Recuadro
x1, y1 = 500, 150
x2, y2 = 1700, 800

# ===============================
# Bucle principal
# ===============================
while True:
    ret, frame = cap.read()

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imshow("Captura", frame)

    key = cv2.waitKey(1)

    if key == ord("s"):

        cropped = frame[y1:y2, x1:x2]
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

        results = zxingcpp.read_barcodes(cropped_rgb)

        if len(results) == 0:
            print("No se encontró ningún código PDF417 dentro del recuadro.")
        else:
            print("\n===== CÓDIGOS DETECTADOS =====")

            for r in results:
                print(f"Formato: {r.format}")

                # Limpieza y extracción
                clean, extracted = parse_pdf417(r.text)

                print("\n--- Texto limpio ---")
                print(clean)

                print("\n--- Datos extraídos ---")
                for key, val in extracted.items():
                    print(f"{key}: {val}")

                print("\n")

    if key == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()


Resolución real: 1920.0 x 1080.0
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro.
No se encontró ningún código PDF417 dentro del recuadro

In [ ]:
import cv2
import zxingcpp
import re

# ===============================
# Función para limpiar y extraer datos de la cédula
# ===============================
def parse_pdf417(text):
    # Quitar caracteres no imprimibles
    clean = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', text)

    # Quitar palabras "NUL" que vienen del decodificador
    clean = clean.replace("NUL", " ")

    data = {}

    # ----------------------------
    # 1. Encontrar SEXO + FECHA
    # ----------------------------
    m = re.search(r'([MF])\s*(\d{8})', clean)
    if m:
        data["sexo"] = m.group(1)
        data["fecha_nac"] = m.group(2)

    # ----------------------------
    # 2. Encontrar PRIMER APELLIDO
    # ----------------------------
    palabras = re.findall(r'\b[A-ZÑÁÉÍÓÚ]{2,}\b', clean)

    if not palabras:
        return clean, data

    apellido1 = palabras[0]
    data["apellido1"] = apellido1

    # ----------------------------
    # 3. Buscar CÉDULA invertida antes del apellido
    # ----------------------------

    pos = clean.find(apellido1)

    antes = clean[:pos].rstrip()

    # Buscar 10 dígitos seguidos justo antes del apellido
    m = re.search(r'(\d{10})\s*$', antes)

    if m:
        invertida = m.group(1)
        cedula = invertida[::-1]  # invertimos los 10 dígitos
        data["cedula"] = cedula
    else:
        data["cedula"] = None

    # ----------------------------
    # 4. APELLIDO2 y NOMBRES
    # ----------------------------
    idx = palabras.index(apellido1)

    if len(palabras) > idx + 1:
        data["apellido2"] = palabras[idx + 1]

    if len(palabras) > idx + 2:
        data["nombre"] = " ".join(palabras[idx + 2:])

    # ----------------------------
    # 5. RH
    # ----------------------------
    m = re.search(r'(A|B|O)[+-]', clean)
    if m:
        data["rh"] = m.group(0)

    return clean, data


# ===============================
# Configuración de cámara
# ===============================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
print("Resolución real:", cap.get(3), "x", cap.get(4))

# Recuadro
x1, y1 = 500, 150
x2, y2 = 1700, 800

# ===============================
# Bucle principal
# ===============================
while True:
    ret, frame = cap.read()

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imshow("Captura", frame)

    key = cv2.waitKey(1)

    if key == ord("s"):

        cropped = frame[y1:y2, x1:x2]
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

        results = zxingcpp.read_barcodes(cropped_rgb)

        if len(results) == 0:
            print("No se encontró ningún código PDF417 dentro del recuadro.")
        else:
            print("\n===== CÓDIGOS DETECTADOS =====")

            for r in results:
                print(f"Formato: {r.format}")

                # Limpieza y extracción
                clean, extracted = parse_pdf417(r.text)

                print("\n--- Texto limpio ---")
                print(clean)

                print("\n--- Datos extraídos ---")
                for key, val in extracted.items():
                    print(f"{key}: {val}")

                print("\n")

    if key == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()

# ===============================
# Configuración de cámara
# ===============================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
print("Resolución real:", cap.get(3), "x", cap.get(4))

# Recuadro
x1, y1 = 500, 150
x2, y2 = 1700, 800

# ===============================
# Bucle principal
# ===============================
while True:
    ret, frame = cap.read()

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imshow("Captura", frame)

    key = cv2.waitKey(1)

    if key == ord("s"):

        cropped = frame[y1:y2, x1:x2]
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

        results = zxingcpp.read_barcodes(cropped_rgb)

        if len(results) == 0:
            print("No se encontró ningún código PDF417 dentro del recuadro.")
        else:
            print("\n===== CÓDIGOS DETECTADOS =====")

            for r in results:
                print(f"Formato: {r.format}")

                # Limpieza y extracción
                clean, extracted = parse_pdf417(r.text)

                print("\n--- Texto limpio ---")
                print(clean)

                print("\n--- Datos extraídos ---")
                for key, val in extracted.items():
                    print(f"{key}: {val}")

                print("\n")

    if key == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()
